## Build Races Dimension

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
%run ../00-common/04-Gold_Helpers

In [0]:
circuits_table = f"{catalog_name}.{silver_schema}.circuits"
races_table = f"{catalog_name}.{silver_schema}.races"

gold_table = f"{catalog_name}.{gold_schema}.dim_races"

In [0]:
from pyspark.sql import functions as f

In [0]:
circuits_df = spark.read.table(circuits_table).filter((f.col("batch_id") == v_batch_id))

races_df = spark.read.table(races_table).filter((f.col("batch_id") == v_batch_id))

In [0]:
joined_df = circuits_df.join(
    races_df, circuits_df.circuit_id == races_df.circuit_id, how="inner"
).select(
    races_df.season, 
    races_df.round, 
    races_df.race_name, 
    races_df.race_date, 
    circuits_df.circuit_name, 
    circuits_df.locality, 
    circuits_df.country
    )

In [0]:
columns = [
    "race_name",
    "race_date",
    "circuit_name",
    "locality",
    "country"
]

write_to_gold(joined_df, gold_table, "trg.season = src.season AND trg.round = src.round", columns)